## Simple use with LCEL

In [28]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
import os

gemini_api_key = os.environ["GEMINI_API_KEY"]
gemini_model = os.environ["GEMINI_MODEL"]

model = GoogleGenerativeAI(model=gemini_model, api_key=gemini_api_key)


In [29]:
from langchain_chroma import Chroma
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings


In [30]:
loader = TextLoader("./data/state_of_the_union.txt")
loaded_document = loader.load()

In [31]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
chunks_of_text = text_splitter.split_documents(loaded_document)

In [32]:
embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

In [33]:
vector_db = Chroma.from_documents(chunks_of_text, embedding_model)

In [34]:
retriever = vector_db.as_retriever()
response = retriever.invoke("what did he say about ketanji brown jackson?")

In [35]:
template = """Answer the question based on the provided context: {context} Question: {question}"""
prompt = ChatPromptTemplate.from_template(template)


In [36]:
def format_doc(docs):
    return "\n\n".join([d.page_content for d in docs])

chain = (
    {"context": retriever | format_doc , "question" : RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
    )

In [37]:
response = chain.invoke("what did he say about ketanji brown jackson?")

In [38]:
response

'Based on the provided context, the speaker said the following about Ketanji Brown Jackson:\n\n* She was nominated by the President 4 days ago to serve on the United States Supreme Court.\n* She is a Circuit Court of Appeals Judge and "one of our nation’s top legal minds."\n* She will "continue Justice Breyer’s legacy of excellence."\n* She is a former top litigator in private practice and a former federal public defender.\n* She comes from a family of public school educators and police officers.\n* She is a consensus builder.\n* Since being nominated, she has received a broad range of support—ranging from the Fraternal Order of Police to former judges appointed by both Democrats and Republicans.'

In [39]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    "Tell me a curious fact about {celebrity}"
)

chain = prompt | model | StrOutputParser()


In [40]:
response = chain.invoke({"celebrity": "Virat Kohli"})

In [41]:
response

'Here is a curious fact about Virat Kohli:\n\nBefore he became a global cricket icon known for his strict fitness regime and vegan diet, **Virat Kohli was a massive "junk food" addict.**\n\nGrowing up, he was completely obsessed with street food—specifically **Ram Chaat** (a famous potato-and-chickpea dish in Delhi) and buttery chicken rolls. He used to eat them almost every day after practice. \n\nThe turning point came in 2012 during the Indian Premier League. Kohli realized he was sluggish, overweight, and not performing at his peak. He recalled looking in the mirror one morning, deciding he needed to change, and completely cut out junk food overnight. He adopted a rigorous workout routine and eventually transitioned to a strict diet (turning vegan in 2018). \n\nThat single lifestyle shift turned him into one of the fittest athletes in the world and fueled his legendary run-chase dominance.'

## Legacy Chain

In [46]:
from langchain_classic.chains import LLMChain
traditionalChain = LLMChain(llm=model, prompt=prompt)

response = traditionalChain.predict(celebrity="Virat Kohli")


In [47]:
response

'Here is a curious fact about Virat Kohli:\n\nBefore he became a global cricket icon known for his strict fitness regime and vegan diet, **Virat Kohli was a self-confessed junk-food addict.** \n\nGrowing up, he was obsessed with *butter chicken*, *rolls*, and fast food. However, a major turning point came in **2012** during an ODI series against Sri Lanka. After a poor performance, he looked in the mirror, realized he was overweight and not performing at his peak, and completely transformed his lifestyle overnight. \n\nHe cut out all junk food, gave up non-vegetarian food in 2018 for health reasons (later shifting to a plant-based diet), and went on to revolutionize the fitness culture of the Indian cricket team.'

In [49]:
for s in chain.stream({"celebrity": "Virat Kohli"}):
    print(s, end="", flush=True)

Here is a curious fact about Virat Kohli:

Before he became one of the greatest batsmen in cricket history, **Virat Kohli was actually rejected by his regional selection committee** when he was a teenager. 

In December 2006, playing in a crucial under-19 match for Delhi against Karnataka, Kohli learned that his father had passed away late the previous night due to a brain stroke. Despite the devastating personal tragedy, the 18-year-old Kohli made the courageous decision to return to the crease the next morning. 

He scored 90 runs, which saved his team from follow-on, and directly after being dismissed, he walked off the field and went straight to his father’s funeral. That defining moment of grit and mental toughness transformed him into the fiercely competitive athlete he is today.

In [51]:
chain.batch([{"celebrity": "Virat Kohli"},{"celebrity": "Sachin Tendulkar"}])

['Here is a curious fact about Virat Kohli:\n\nDespite being one of the greatest and most aggressive modern-era batsmen in the world, **Virat Kohli has never scored a Test match century using a brand-new bat.** \n\nAccording to his childhood coach, Rajkumar Sharma, Kohli has a quirky superstition: he prefers to use bats that are already "broken in" and have a good feel to them, rather than brand-new ones straight out of the box. He often knocks them in extensively or keeps using a bat even after it has developed cracks, sometimes just taping it up, because he feels most comfortable and confident with a well-used blade.',
 "Here is a curious and little-known fact about Sachin Tendulkar:\n\n**He actually started his international career playing for Pakistan!**\n\nDuring a practice match between India and Pakistan at the Brabourne Stadium in Mumbai on **January 20, 1987**, Pakistan was fielding and found themselves short of a few players due to injuries and rest. \n\nKapil Dev, the Indian